# Notebook 04: Beta Diversity Analysis

Between-sample diversity analysis using Bray-Curtis dissimilarity, PCoA ordination,
PERMANOVA for group separation, and (optionally) UMAP non-linear embedding.
Datasets: Colorectal (PRJNA290926), Breast (PRJNA658160), Prostate (PRJNA1298576).

In [ ]:
# ============================================================
# CELL 1 — Import Libraries
# ============================================================
# Load all the tools we need before starting the analysis.

import pandas as pd          # for working with tables (DataFrames)
import numpy as np           # for fast math and array operations
import matplotlib            # the base plotting library
matplotlib.use('Agg')        # save plots to files instead of displaying pop-up windows
import matplotlib.pyplot as plt   # the easy-to-use matplotlib interface
import seaborn as sns        # for prettier statistical charts
from scipy.spatial.distance import pdist, squareform, cdist  # distance calculation functions
# pdist: pairwise distances from a 2D array (returns flattened upper triangle)
# squareform: converts the flattened result into a full square matrix
# cdist: computes distances between two sets of samples — we use this for Bray-Curtis
from scipy.linalg import eigh  # eigenvalue decomposition for symmetric matrices (used in PCoA)
import warnings
warnings.filterwarnings('ignore')  # suppress minor warnings so output is clean
import os                    # file system operations (paths, folders)

print("Imports complete.")

In [ ]:
# ============================================================
# CELL 2 — Paths and Load Data
# ============================================================
# Set the project root folder path (using a raw string r"..." to handle backslashes on Windows).
# Then load the combined genus-level abundance table and metadata saved by Notebook 02.

BASE_DIR = r"c:\MyProjects\Project-Proposal\PrivateCoach\In Progress\Project-BioInformatics\Projects\cancer-microbiome\Final_Solution"
RESULTS_DIR = os.path.join(BASE_DIR, "Results")   # folder where CSVs were saved by earlier notebooks
FIGURES_DIR = os.path.join(BASE_DIR, "Figures")   # folder where we'll save our plots

os.makedirs(RESULTS_DIR, exist_ok=True)  # create Results/ if missing
os.makedirs(FIGURES_DIR, exist_ok=True)  # create Figures/ if missing

# Load the combined genus-level abundance table (all 3 cancer types, 618 samples, 259 genera).
# This was created by Notebook 02 after taxonomic harmonization and CLR-free filtering.
# Values here are raw relative abundances (fractions 0–1), NOT CLR-transformed.
abund = pd.read_csv(os.path.join(RESULTS_DIR, "abund_combined_genus.csv"), index_col=0)

# Load the sample metadata (cancer_type and condition labels for each sample)
meta = pd.read_csv(os.path.join(RESULTS_DIR, "meta_combined.csv"), index_col=0)

# Make sure both tables have the same set of samples (rows) in the same order.
# index.intersection() finds sample IDs that exist in both tables.
common_idx = abund.index.intersection(meta.index)
abund = abund.loc[common_idx]  # filter abundance to only keep shared samples
meta  = meta.loc[common_idx]   # filter metadata to match

print(f"Abundance matrix shape : {abund.shape}  (samples x genera)")   # expect (618, 259)
print(f"Metadata shape         : {meta.shape}")                         # expect (618, 2)
print(f"Cancer types           : {meta['cancer_type'].value_counts().to_dict()}")   # samples per type
print(f"Conditions             : {meta['condition'].value_counts().to_dict()}")     # cancer vs healthy

In [ ]:
# ============================================================
# CELL 3 — Compute Bray-Curtis Dissimilarity Matrix
# ============================================================
# Bray-Curtis dissimilarity measures how DIFFERENT two samples are in bacterial composition.
# Value 0 = identical communities; Value 1 = completely different communities.
#
# Formula: BC(A, B) = 1 - 2 * sum(min(A_i, B_i)) / (sum(A) + sum(B))
# where A_i and B_i are the abundances of species i in samples A and B.
#
# We use cdist (cross-distance) from scipy to compute all pairs at once efficiently.

print("Computing Bray-Curtis dissimilarity matrix ...")

# cdist computes the distance between EVERY pair of rows in the abundance matrix.
# metric='braycurtis' tells it to use the Bray-Curtis formula instead of Euclidean.
# Result shape: (618, 618) — entry [i, j] is the distance between sample i and sample j.
bc_dist = cdist(abund.values, abund.values, metric='braycurtis')

# Wrap the numpy array in a DataFrame so rows and columns are labelled with sample IDs
bc_matrix = pd.DataFrame(bc_dist, index=abund.index, columns=abund.index)

print(f"Bray-Curtis matrix shape: {bc_matrix.shape}")
# np.triu_indices_from extracts the upper triangle (avoiding duplicate pairs and self-comparisons)
print(f"Distance range           : {bc_dist.min():.4f} – {bc_dist.max():.4f}")
print(f"Mean distance            : {bc_dist[np.triu_indices_from(bc_dist, k=1)].mean():.4f}")

# Save the full distance matrix — it will be used by PERMANOVA and the box plot below
bc_out = os.path.join(RESULTS_DIR, 'bray_curtis_matrix.csv')
bc_matrix.to_csv(bc_out)
print(f"Saved Bray-Curtis matrix : {bc_out}")

In [ ]:
# ============================================================
# CELL 4 — PCoA: Principal Coordinates Analysis
# ============================================================
# PCoA is like a "map" of our 618 samples.
# Each sample is a point; samples with similar bacteria end up close together.
# We go from a 618×618 distance table to a 2D (or 3D) scatterplot we can visualize.
#
# The math (Gower 1966):
#  1. Square every distance: D² 
#  2. Double-center the matrix: B = -½ × H × D² × H   (H removes the row/column means)
#  3. Take the eigendecomposition of B: B = VΛVᵀ
#  4. Keep only the positive eigenvalues; the coordinates are V × √Λ
#  5. The fraction of variance explained by axis k = λ_k / Σλ

def pcoa(dist_matrix):
    """
    Classical metric MDS / PCoA from a symmetric distance matrix.

    Steps
    -----
    1. Square the distances.
    2. Double-center: B = -0.5 * H D^2 H, where H = I - (1/n) 11^T.
    3. Eigen-decompose B; keep positive eigenvalues only.
    4. Coordinates = eigenvectors * sqrt(eigenvalues).
    """
    n = len(dist_matrix)                # number of samples
    D2 = dist_matrix.values ** 2        # square every distance

    # Build the centering matrix H = I - (1/n) * ones_matrix
    H = np.eye(n) - np.ones((n, n)) / n  # H removes row/column means
    B = -0.5 * H @ D2 @ H               # double-centering; @ = matrix multiplication

    # eigh is faster than eig for symmetric matrices; returns eigenvalues sorted ascending
    eigenvalues, eigenvectors = eigh(B)

    # Sort eigenvalues (and their vectors) in DESCENDING order (largest variance first)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues  = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    # Discard tiny/negative eigenvalues (numerical noise from floating-point)
    pos_mask    = eigenvalues > 1e-10
    eigenvalues = eigenvalues[pos_mask]
    eigenvectors = eigenvectors[:, pos_mask]

    # Each column of 'coords' is one PCoA axis; values are the sample coordinates
    coords = eigenvectors * np.sqrt(eigenvalues)   # scale by √eigenvalue

    # Fraction of total variance captured by each axis (expressed as percent)
    explained = eigenvalues / eigenvalues.sum() * 100

    return coords, explained


print("Running PCoA ...")
coords, explained = pcoa(bc_matrix)  # run PCoA on the 618×618 Bray-Curtis matrix

# Build a DataFrame with the first 3 axes (PC1, PC2, PC3)
n_axes    = min(3, coords.shape[1])              # take up to 3 axes
col_names = [f'PC{i+1}' for i in range(n_axes)] # column names: PC1, PC2, PC3
pcoa_df   = pd.DataFrame(coords[:, :n_axes], index=bc_matrix.index, columns=col_names)

# Join metadata so each row knows its cancer_type and condition
pcoa_df = pcoa_df.join(meta[['cancer_type', 'condition']])

print(f"PCoA complete. Total positive axes: {len(explained)}")
for i in range(min(5, len(explained))):
    print(f"  PC{i+1}: {explained[i]:.2f}% variance explained")  # how much each axis explains

In [ ]:
# ============================================================
# CELL 5 — Figure 2: PCoA Scatter Plot
# ============================================================
# This scatter plot shows all 618 samples as dots on a 2D "map".
# Samples from the same cancer type should cluster together if their
# bacterial communities are similar.
# Shape (circle vs triangle) shows whether a sample is Cancer or Healthy.

# Colour dictionary: each cancer type gets a distinctive colour
PALETTE = {
    'Colorectal': '#2196F3',  # blue
    'Breast':     '#E91E63',  # pink
    'Prostate':   '#4CAF50',  # green
}
MARKERS = {'Cancer': 'o', 'Healthy': '^'}  # filled circle for Cancer, triangle for Healthy

fig, ax = plt.subplots(figsize=(8, 6))

# Loop through each cancer type, then each condition, and draw the dots
for ct, ct_grp in pcoa_df.groupby('cancer_type'):      # group by cancer type
    for cond, cond_grp in ct_grp.groupby('condition'): # group by Cancer / Healthy
        marker = MARKERS.get(cond, 'o')                # pick circle or triangle
        label  = f"{ct} ({cond})"                      # legend label, e.g. "Breast (Cancer)"
        ax.scatter(
            cond_grp['PC1'], cond_grp['PC2'],           # x and y coordinates
            c=PALETTE.get(ct, 'grey'),                  # dot color
            marker=marker,
            s=40,        # dot size
            alpha=0.7,   # transparency (0=invisible, 1=opaque)
            label=label,
            edgecolors='white', linewidths=0.3,         # thin white outline around dots
        )

# Label axes with the % variance captured by each axis
ax.set_xlabel(f"PC1 ({explained[0]:.1f}% variance)", fontsize=12)
ax.set_ylabel(f"PC2 ({explained[1]:.1f}% variance)", fontsize=12)
ax.set_title('PCoA of Bray-Curtis Dissimilarity (All Cancer Types)', fontsize=13, fontweight='bold')

# Place the legend outside the plot area to avoid covering dots
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9, frameon=True)

plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, 'fig02_pcoa_bray_curtis.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved: {fig_path}")

In [ ]:
# ============================================================
# CELL 6 — PERMANOVA: Permutational Multivariate Analysis of Variance
# ============================================================
# PERMANOVA answers: "Are the microbiomes of our cancer groups statistically DIFFERENT?"
# It's like an ANOVA but for distance matrices instead of raw measurements.
#
# How it works:
#   1. Calculate the real F-statistic (ratio of between-group to within-group spread)
#   2. Shuffle the group labels randomly 999 times and recalculate F each time
#   3. p-value = fraction of shuffled F's that are >= the real F
#      A small p-value (<0.05) means the real groups ARE more separated than by chance.
#
# R² = fraction of total variance explained by group membership.

def permanova(dist_matrix, groups, n_permutations=999, random_state=42):
    """
    Simplified PERMANOVA for group separation testing.

    Parameters
    ----------
    dist_matrix    : pd.DataFrame, symmetric distance matrix
    groups         : pd.Series, group labels aligned to dist_matrix index
    n_permutations : int, number of permutations for p-value
    random_state   : int, for reproducibility

    Returns
    -------
    F_obs  : float, observed F-statistic
    R2_obs : float, proportion of variance explained by groups
    p_value: float, permutation p-value
    """
    rng = np.random.default_rng(random_state)  # reproducible random number generator

    dist_sq = squareform(dist_matrix.values)   # convert square matrix to condensed upper-triangle form
    labels  = groups.values                    # group labels as a numpy array
    n       = len(labels)

    def compute_F_R2(labels, dist_sq_full):
        """Compute F-statistic and R² from the full n×n distance matrix."""
        n = len(labels)
        SS_total = np.sum(dist_sq_full ** 2) / (2 * n)   # total spread (sum of squared distances)
        unique_grps = np.unique(labels)
        k = len(unique_grps)                               # number of groups
        SS_within = 0.0
        for g in unique_grps:
            mask = labels == g          # mask for samples in group g
            n_g  = mask.sum()
            if n_g > 1:
                d_g = dist_sq_full[np.ix_(mask, mask)]    # distances within group g
                SS_within += np.sum(d_g ** 2) / (2 * n_g) # within-group spread
        SS_between = SS_total - SS_within   # between-group spread
        if SS_within == 0 or (n - k) == 0:
            return np.nan, np.nan
        F  = (SS_between / (k - 1)) / (SS_within / (n - k))  # F-statistic formula
        R2 = SS_between / SS_total                             # R² = fraction explained by groups
        return F, R2

    dist_full = dist_matrix.values
    F_obs, R2_obs = compute_F_R2(labels, dist_full)   # observed statistic on real data

    # Permutation test: shuffle labels, recompute F, count how many times F_perm >= F_obs
    F_perm = []
    for _ in range(n_permutations):
        perm_labels = rng.permutation(labels)            # randomly shuffle the group labels
        F_p, _      = compute_F_R2(perm_labels, dist_full)
        if not np.isnan(F_p):
            F_perm.append(F_p)

    # p-value: +1 in numerator and denominator to include the observed result itself (conservative)
    p_value = (np.sum(np.array(F_perm) >= F_obs) + 1) / (len(F_perm) + 1)
    return F_obs, R2_obs, p_value


print("Running PERMANOVA (999 permutations) ...")
F, R2, p_val = permanova(bc_matrix, meta['cancer_type'], n_permutations=999)
print(f"PERMANOVA across cancer types:")
print(f"  F    = {F:.3f}")    # large F = strong group separation
print(f"  R²   = {R2:.3f}")   # fraction of total variation explained by cancer type
print(f"  p    = {p_val:.3f} {'(significant)' if p_val < 0.05 else '(not significant)'}")

# Pairwise PERMANOVA: test each pair of cancer types separately
from itertools import combinations

cancer_types = sorted(meta['cancer_type'].unique())
perm_rows = [{
    'comparison': 'All groups',
    'F': round(F, 4),
    'R2': round(R2, 4),
    'p_value': round(p_val, 4),
    'significant': p_val < 0.05,
}]

print("\nPairwise PERMANOVA:")
for ct1, ct2 in combinations(cancer_types, 2):
    mask     = meta['cancer_type'].isin([ct1, ct2])   # keep only the two groups being compared
    sub_meta = meta[mask]
    sub_bc   = bc_matrix.loc[sub_meta.index, sub_meta.index]  # subset distance matrix
    F_p, R2_p, p_p = permanova(sub_bc, sub_meta['cancer_type'], n_permutations=999)
    label = f"{ct1} vs {ct2}"
    perm_rows.append({
        'comparison': label,
        'F': round(F_p, 4),
        'R2': round(R2_p, 4),
        'p_value': round(p_p, 4),
        'significant': p_p < 0.05,
    })
    print(f"  {label}: F={F_p:.3f}, R²={R2_p:.3f}, p={p_p:.3f} {'*' if p_p < 0.05 else ''}")

perm_df = pd.DataFrame(perm_rows)
perm_out = os.path.join(RESULTS_DIR, 'permanova_results.csv')
perm_df.to_csv(perm_out, index=False)
print(f"\nSaved PERMANOVA results: {perm_out}")

In [ ]:
# ============================================================
# CELL 7 — UMAP: Non-linear Dimensionality Reduction (optional)
# ============================================================
# UMAP stands for Uniform Manifold Approximation and Projection.
# Like PCoA, it makes a 2D "map" of samples, but it can reveal CURVED clusters
# that PCoA (which is linear) might miss.
# It requires the extra package 'umap-learn' — if it's not installed, we skip this cell.

umap_available = False   # flag to track whether UMAP ran successfully
umap_df = None           # will hold the UMAP coordinates if we succeed

try:
    from umap import UMAP   # try to import UMAP — fails if umap-learn is not installed
    print("umap-learn found. Running UMAP on Bray-Curtis distance matrix ...")

    reducer = UMAP(
        n_components=2,       # reduce to 2 dimensions for plotting
        random_state=42,      # fixed seed for reproducibility
        metric='precomputed', # use our pre-computed Bray-Curtis matrix instead of raw data
        n_neighbors=15,       # how many nearby samples to consider when building the map
        min_dist=0.1,         # minimum distance between points in the 2D map
    )

    # fit_transform learns the UMAP mapping and applies it; input is our distance matrix
    umap_coords = reducer.fit_transform(bc_matrix.values)  # output: (618, 2) array

    # Wrap coordinates in a DataFrame and attach metadata labels
    umap_df = pd.DataFrame(
        umap_coords,
        columns=['UMAP1', 'UMAP2'],  # the two compressed dimensions
        index=bc_matrix.index,        # preserve sample IDs as row labels
    )
    umap_df = umap_df.join(meta[['cancer_type', 'condition']])  # add cancer type and condition columns

    umap_available = True
    print("UMAP complete.")
    print(umap_df.head())   # preview first 5 rows

except ImportError:
    # This error only fires if umap-learn is not installed
    print("umap-learn not installed. Skipping UMAP.")
    print("Install with: pip install umap-learn")

In [ ]:
# ============================================================
# CELL 8 — Figure 3: UMAP Plot
# ============================================================
# If UMAP ran successfully, draw the scatter plot.
# If not, create a placeholder image with instructions to install umap-learn.

PALETTE = {
    'Colorectal': '#2196F3',  # blue
    'Breast':     '#E91E63',  # pink
    'Prostate':   '#4CAF50',  # green
}
MARKERS = {'Cancer': 'o', 'Healthy': '^'}

if umap_available and umap_df is not None:
    # --- UMAP is available: draw the real scatter plot ---
    fig, ax = plt.subplots(figsize=(8, 6))

    for ct, ct_grp in umap_df.groupby('cancer_type'):      # group by cancer type
        for cond, cond_grp in ct_grp.groupby('condition'): # group by Cancer vs Healthy
            marker = MARKERS.get(cond, 'o')
            label  = f"{ct} ({cond})"
            ax.scatter(
                cond_grp['UMAP1'], cond_grp['UMAP2'],  # x and y are the two UMAP axes
                c=PALETTE.get(ct, 'grey'),
                marker=marker,
                s=40, alpha=0.7,
                label=label,
                edgecolors='white', linewidths=0.3,
            )

    ax.set_xlabel('UMAP1', fontsize=12)  # UMAP axes don't have a direct physical meaning
    ax.set_ylabel('UMAP2', fontsize=12)
    ax.set_title('UMAP of Bray-Curtis Dissimilarity (All Cancer Types)', fontsize=13, fontweight='bold')
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9, frameon=True)

    plt.tight_layout()
    fig_path = os.path.join(FIGURES_DIR, 'fig03_umap.png')
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {fig_path}")

else:
    # --- UMAP not available: save a placeholder image ---
    print("UMAP not available — skipping Figure 3. Install umap-learn and re-run Cell 7.")
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.text(
        0.5, 0.5,
        "UMAP not available.\nInstall umap-learn:\n  pip install umap-learn",
        transform=ax.transAxes,
        ha='center', va='center', fontsize=14,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8),
    )
    ax.set_axis_off()
    ax.set_title('UMAP — Not Available', fontsize=13)
    plt.tight_layout()
    fig_path = os.path.join(FIGURES_DIR, 'fig03_umap.png')
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved placeholder: {fig_path}")

In [ ]:
# ============================================================
# CELL 9 — Within-Group vs Between-Group Bray-Curtis Box Plot
# ============================================================
# Idea: if bacteria really differ between cancer types, then
# two samples from DIFFERENT types should be more dissimilar
# (bigger Bray-Curtis distance) than two samples from the SAME type.

PALETTE = {
    'Colorectal': '#2196F3',  # blue
    'Breast':     '#E91E63',  # pink
    'Prostate':   '#4CAF50',  # green
}

cancer_types = sorted(meta['cancer_type'].unique())   # ['Breast', 'Colorectal', 'Prostate']

# Build a list of (distance, label) for every unique pair of samples (upper triangle)
dist_records = []
bc_vals  = bc_matrix.values       # extract the numpy array for speed
idx_list = list(bc_matrix.index)  # sample ID list
n = len(idx_list)

from itertools import combinations   # generates pairs efficiently

for i, j in combinations(range(n), 2):   # every unique pair (i < j so no duplicates)
    ct_i = meta.loc[idx_list[i], 'cancer_type']   # cancer type of sample i
    ct_j = meta.loc[idx_list[j], 'cancer_type']   # cancer type of sample j
    d    = bc_vals[i, j]                           # Bray-Curtis distance between them

    if ct_i == ct_j:
        label = f"{ct_i}\n(within)"    # same cancer type → "within-group"
    else:
        pair  = ' vs '.join(sorted([ct_i, ct_j]))  # sort so the label is consistent
        label = f"{pair}\n(between)"  # different cancer types → "between-group"

    dist_records.append({'comparison': label, 'distance': d})

dist_long = pd.DataFrame(dist_records)   # tidy DataFrame for seaborn plotting

# Decide column order: within-group boxes first, then between-group
within_labels  = [f"{ct}\n(within)" for ct in cancer_types]
between_labels = [f"{' vs '.join(sorted([ct1, ct2]))}\n(between)"
                  for ct1, ct2 in combinations(cancer_types, 2)]
order = [lbl for lbl in (within_labels + between_labels)
         if lbl in dist_long['comparison'].unique()]

# Map each label to a color
within_colors = {f"{ct}\n(within)": PALETTE[ct] for ct in cancer_types}
color_map     = {lbl: within_colors.get(lbl, '#9E9E9E') for lbl in order}
palette_list  = [color_map[lbl] for lbl in order]

fig, ax = plt.subplots(figsize=(12, 6))

sns.boxplot(
    data=dist_long,
    x='comparison', y='distance',
    order=order,
    palette=palette_list,
    width=0.55,
    flierprops=dict(marker='.', markersize=2, alpha=0.3),   # tiny dots for outliers
    ax=ax,
)

ax.set_xlabel('Comparison', fontsize=12)
ax.set_ylabel('Bray-Curtis Dissimilarity', fontsize=12)
ax.set_title('Within- vs Between-Group Beta Diversity', fontsize=13, fontweight='bold')
ax.tick_params(axis='x', labelsize=9)

# Dashed line at the global median for reference
global_median = dist_long['distance'].median()
ax.axhline(global_median, color='black', linestyle='--', linewidth=0.8, alpha=0.5,
           label=f'Global median = {global_median:.3f}')
ax.legend(fontsize=9)

plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, 'fig04c_beta_diversity_groups.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved: {fig_path}")

# Print summary: higher between-group than within-group confirms real biological signal
print("\nMedian Bray-Curtis distances (higher = more different):")
print(dist_long.groupby('comparison')['distance'].median().round(4).to_string())

In [ ]:
# ============================================================
# CELL 10 — Save PCoA and UMAP Coordinates to CSV
# ============================================================
# These CSV files let other notebooks (or future analyses)
# re-use the 2D coordinates without re-running the heavy math.

# --- Save PCoA coordinates ---
pcoa_out = os.path.join(RESULTS_DIR, 'pcoa_coordinates.csv')
pcoa_df.to_csv(pcoa_out)   # index=True by default (sample IDs are the row index)
print(f"Saved PCoA coordinates : {pcoa_out}")
print(f"  Shape: {pcoa_df.shape}  (samples x PC axes + metadata columns)")

# --- Save UMAP coordinates (only if UMAP ran successfully) ---
if umap_available and umap_df is not None:
    umap_out = os.path.join(RESULTS_DIR, 'umap_coordinates.csv')
    umap_df.to_csv(umap_out)
    print(f"Saved UMAP coordinates : {umap_out}")
    print(f"  Shape: {umap_df.shape}  (samples x UMAP1, UMAP2 + metadata)")
else:
    print("UMAP coordinates not saved (umap-learn not available or UMAP did not run).")

# --- Verify all output files exist ---
print("\nFinal file check:")
all_files = [
    ('Results', 'bray_curtis_matrix.csv'),
    ('Results', 'permanova_results.csv'),
    ('Results', 'pcoa_coordinates.csv'),
    ('Figures', 'fig02_pcoa_bray_curtis.png'),
    ('Figures', 'fig03_umap.png'),
    ('Figures', 'fig04c_beta_diversity_groups.png'),
]
for folder, fname in all_files:
    fpath  = os.path.join(BASE_DIR, folder, fname)
    status = '[OK]' if os.path.isfile(fpath) else '[MISSING]'
    print(f"  {status}  {folder}/{fname}")

print("\nNotebook 04 — Beta Diversity — COMPLETE.")